# Lab 5: Inheritance and Abstraction
**Course:** DATA 2402  
**Topic:** Abstract Base Classes (ABC), Polymorphic Class Hierarchies, and Constraint-Checked Combat Simulation

### Demonstrated Components:
1. **Task 1 & 2:** Weapon inheritance hierarchy (`Weapon` -> `MeleeWeapon`, `RangedWeapon` -> `MagicWeapon`).
2. **Task 3:** Character inheritance hierarchy (`Character` -> `Fighter`, `Mage`).
3. **Task 4:** `attack()` function with constraint checking and custom exception handling.

In [ ]:
import sys
import os

# Ensure notebook looks in the current folder for characters.py
sys.path.append(os.path.abspath('.'))

from characters1 import Fighter, Mage, MeleeWeapon, RangedWeapon, MagicWeapon

# Instantiate characters
aragorn = Fighter("Aragorn", health=120, speed=10)
gandalf = Mage("Gandalf", health=80, mana=150)
orc = Fighter("Orc Grunt", health=60, speed=8)

# Instantiate weapons
sword = MeleeWeapon("Broadsword", damage=25)
bow = RangedWeapon("Longbow", damage=20, range=50)
staff = MagicWeapon("Staff of Light", damage=45, range=30, mana_cost=25)

print("Characters:")
print(f" - {aragorn}")
print(f" - {gandalf}")
print(f" - {orc}")

print("\nWeapons:")
print(f" - {sword}")
print(f" - {bow}")
print(f" - {staff}")

Characters:
 - Fighter,Aragorn,120,10
 - Mage,Gandalf,80,150
 - Fighter,Orc Grunt,60,8

Weapons:
 - MeleeWeapon,Broadsword,25
 - RangedWeapon,Longbow,20,50
 - MagicWeapon,Staff of Light,45,30,25


In [ ]:
def attack(attacker, weapon, target, distance: int) -> None:
    if distance < 0:
        raise ValueError("Attack distance cannot be negative.")
    
    # 1. Only Mages can use a MagicWeapon
    if isinstance(weapon, MagicWeapon) and not isinstance(attacker, Mage):
        raise TypeError(f"Illegal Weapon: Only Mages can use {weapon.name}, but {attacker.name} is a {type(attacker).__name__}.")

    # 2. Only weapons with range can be used at distance > 0
    if not hasattr(weapon, "range") and distance > 0:
        raise ValueError(f"Out of Reach: {weapon.name} has no range attribute and cannot attack at distance {distance} > 0.")

    # 3. Weapons with range cannot exceed that range
    if hasattr(weapon, "range") and distance > weapon.range:
        raise ValueError(f"Out of Range: Target distance ({distance}) exceeds {weapon.name}'s range ({weapon.range}).")

    target.health -= weapon.damage
    print(f"SUCCESS: {attacker.name} attacked {target.name} with {weapon.name} at distance {distance}m. Deal {weapon.damage} dmg! (Target HP: {target.health})")

## Verification: Normal & Valid Cases
Demonstrating valid combat interactions across melee, ranged, and magic weapons.

In [ ]:
# Test 1: Melee attack at distance 0 (Valid)
attack(attacker=aragorn, weapon=sword, target=orc, distance=0)

# Test 2: Ranged attack within range (distance 35 <= 50) (Valid)
attack(attacker=aragorn, weapon=bow, target=orc, distance=35)

# Test 3: Mage using MagicWeapon within range (distance 20 <= 30) (Valid)
attack(attacker=gandalf, weapon=staff, target=orc, distance=20)

SUCCESS: Aragorn attacked Orc Grunt with Broadsword at distance 0m. Deal 25 dmg! (Target HP: 35)
SUCCESS: Aragorn attacked Orc Grunt with Longbow at distance 35m. Deal 20 dmg! (Target HP: 15)
SUCCESS: Gandalf attacked Orc Grunt with Staff of Light at distance 20m. Deal 45 dmg! (Target HP: -30)


## Verification: Constraint Violations & Error Handling
Demonstrating that all required constraints trigger exceptions appropriately.

In [4]:
# Error Case 1: Non-Mage using MagicWeapon
try:
    print("\n[Attempt 1: Fighter using MagicWeapon]")
    attack(attacker=aragorn, weapon=staff, target=orc, distance=10)
except TypeError as e:
    print(f"Caught Expected Exception: {e}")

# Error Case 2: Melee weapon used at distance > 0
try:
    print("\n[Attempt 2: Melee weapon at distance > 0]")
    attack(attacker=aragorn, weapon=sword, target=orc, distance=5)
except ValueError as e:
    print(f"Caught Expected Exception: {e}")

# Error Case 3: Ranged weapon used past maximum range
try:
    print("\n[Attempt 3: Attack beyond maximum weapon range]")
    attack(attacker=aragorn, weapon=bow, target=orc, distance=75)
except ValueError as e:
    print(f"Caught Expected Exception: {e}")


[Attempt 1: Fighter using MagicWeapon]
Caught Expected Exception: Illegal Weapon: Only Mages can use Staff of Light, but Aragorn is a Fighter.

[Attempt 2: Melee weapon at distance > 0]
Caught Expected Exception: Out of Reach: Broadsword has no range attribute and cannot attack at distance 5 > 0.

[Attempt 3: Attack beyond maximum weapon range]
Caught Expected Exception: Out of Range: Target distance (75) exceeds Longbow's range (50).


In [5]:
# Test persistence requirement for weapons
sword.save("saved_weapons.txt")
bow.save("saved_weapons.txt")
staff.save("saved_weapons.txt")

with open("saved_weapons.txt", "r") as f:
    print("Contents of saved_weapons.txt:")
    print(f.read())

Contents of saved_weapons.txt:
MeleeWeapon,Broadsword,25
RangedWeapon,Longbow,20,50
MagicWeapon,Staff of Light,45,30,25

